# **1. Subir el dataset**

In [31]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

In [32]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [33]:
import pandas as pd

file_path = '/content/drive/MyDrive/Proyecto IA/Data/feeling.csv'

# Load the dataset into a pandas DataFrame
try:
    df = pd.read_csv(file_path)
    print("Dataset loaded successfully!")
    display(df.head())
except FileNotFoundError:
    print(f"Error: The file was not found at {file_path}")
except Exception as e:
    print(f"An error occurred: {e}")

Dataset loaded successfully!


,text,label
0,i feel awful about it too because it s my job ...,0
1,im alone i feel awful,0
2,ive probably mentioned this before but i reall...,1
3,i was feeling a little low few days back,0
4,i beleive that i am much more sensitive to oth...,2


# **2. Transformación de los datos**

## Desbalancear el df

In [34]:
import pandas as pd

counts = df["label"].value_counts().sort_values(ascending=False)
perc   = (counts / counts.sum() * 100).round(2)

balance_tbl = pd.DataFrame({"count": counts, "percent": perc})
display(balance_tbl)
print("Total muestras:", counts.sum())
print("Clase mayoritaria (%):", perc.iloc[0], " | Clase minoritaria (%):", perc.iloc[-1])
print("Ratio mayoritaria/minoritaria:", round(counts.iloc[0]/counts.iloc[-1], 2))

,count,percent
label,,
1,141067,33.84
0,121187,29.07
3,57317,13.75
4,47712,11.45
2,34554,8.29
5,14972,3.59


Total muestras: 416809
Clase mayoritaria (%): 33.84  | Clase minoritaria (%): 3.59
Ratio mayoritaria/minoritaria: 9.42


In [35]:
df = df[~df['label'].isin([2, 5])]
df

,text,label
0,i feel awful about it too because it s my job ...,0
1,im alone i feel awful,0
2,ive probably mentioned this before but i reall...,1
3,i was feeling a little low few days back,0
6,i am one of those people who feels like going ...,1
...,...,...
416804,that was what i felt when i was finally accept...,1
416805,i take every day as it comes i m just focussin...,4
416806,i just suddenly feel that everything was fake,0
416807,im feeling more eager than ever to claw back w...,1


In [36]:
unique_labels = sorted(df['label'].unique())
mapping = {old: new for new, old in enumerate(unique_labels)}
df['label'] = df['label'].map(mapping)
df

,text,label
0,i feel awful about it too because it s my job ...,0
1,im alone i feel awful,0
2,ive probably mentioned this before but i reall...,1
3,i was feeling a little low few days back,0
6,i am one of those people who feels like going ...,1
...,...,...
416804,that was what i felt when i was finally accept...,1
416805,i take every day as it comes i m just focussin...,3
416806,i just suddenly feel that everything was fake,0
416807,im feeling more eager than ever to claw back w...,1


In [37]:
counts = df["label"].value_counts().sort_values(ascending=False)
perc = (counts / counts.sum() * 100).round(2)
balance_tbl = pd.DataFrame({"count": counts, "percent": perc})

display(balance_tbl)

,count,percent
label,,
1,141067,38.41
0,121187,33.00
2,57317,15.61
3,47712,12.99


# **3. Separación del DataFrame**

In [38]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    random_state=42,
    stratify=df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["label"]
)

print(f"Train: {len(train_df)}, Validation: {len(val_df)}, Test: {len(test_df)}")

print("\nDistribución de clases:")
print(train_df["label"].value_counts(normalize=True).sort_index().round(3))
print(val_df["label"].value_counts(normalize=True).sort_index().round(3))
print(test_df["label"].value_counts(normalize=True).sort_index().round(3))


Train: 257098, Validation: 55092, Test: 55093

Distribución de clases:
label
0    0.330
1    0.384
2    0.156
3    0.130
Name: proportion, dtype: float64
label
0    0.330
1    0.384
2    0.156
3    0.130
Name: proportion, dtype: float64
label
0    0.330
1    0.384
2    0.156
3    0.130
Name: proportion, dtype: float64


In [39]:
muestra = train_df.sample(10, random_state=42)

for _, row in muestra.iterrows():
    print("Question:", row["text"])
    print("Label:", row["label"])

Question: i feel passionate about and i want others to include it in their lives too
Label: 1
Question: im cute i really smile o thats what life is about o making other people feel good o and that doesnt mean you have to buy them something
Label: 1
Question: i feel so honoured to have actually witnessed it with my own eyes
Label: 1
Question: i if we were feeling dangerous
Label: 2
Question: i feel that i am the most lucky man in the world
Label: 1
Question: i still feel distraught and crushed
Label: 3
Question: i feel strangely neglectful for not doing something that i do every year at this time
Label: 0
Question: i can say is that i just feel selfish and like a failure
Label: 2
Question: i was feeling frightened at first now im much more confident
Label: 3
Question: i really that worried about them how badly do i feel for them am i really so eager about freeing them from my uncle s selfishness
Label: 1


In [40]:
class_names = ["sadness","joy","love","anger","fear","surprise"]

class_to_index = {name: i for i, name in enumerate(class_names)}
index_to_class = {i: name for i, name in enumerate(class_names)}

print(class_to_index)

{'sadness': 0, 'joy': 1, 'love': 2, 'anger': 3, 'fear': 4, 'surprise': 5}


# **4. Vectorización del texto**

In [41]:
MAX_SEQUENCE_LENGTH = 100
VOCAB_SIZE = 10000

int_vectorize_layer = TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_SEQUENCE_LENGTH)

In [42]:
texts = train_df["text"].astype(str).values
labels = train_df["label"].values

texts_test = test_df["text"].astype(str).values
labels_test = test_df["label"].values

texts_val = val_df["text"].astype(str).values
labels_val = val_df["label"].values

In [43]:
train_ds = tf.data.Dataset.from_tensor_slices((texts, labels)).batch(128)
test_ds = tf.data.Dataset.from_tensor_slices((texts_test, labels_test)).batch(128)
val_ds = tf.data.Dataset.from_tensor_slices((texts_val, labels_val)).batch(128)

In [44]:
train_text = train_ds.map(lambda text, label: text)
binary_vectorize_layer.adapt(train_text)
int_vectorize_layer.adapt(train_text)

In [45]:
def int_vectorize_text(text, label):
  text = tf.expand_dims(text, -1)
  return int_vectorize_layer(text), label

In [46]:
text_batch, label_batch = next(iter(train_ds))
first_question, first_label = text_batch[3], label_batch[3]
print("Text", first_question)
print("Label", first_label)

Text tf.Tensor(b'i really feel that i missed out', shape=(), dtype=string)
Label tf.Tensor(0, shape=(), dtype=int64)


In [47]:
print("Original text:", first_question.numpy()[:200])
print("'int' vectorized question:",
      int_vectorize_text(first_question, first_label)[0])

Original text: b'i really feel that i missed out'
'int' vectorized question: tf.Tensor(
[[  2  41   3   9   2 387  48   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0]], shape=(1, 100), dtype=int64)


In [48]:
print("1289 ---> ", int_vectorize_layer.get_vocabulary()[1289])
print("313 ---> ", int_vectorize_layer.get_vocabulary()[313])
print("Vocabulary size: {}".format(len(int_vectorize_layer.get_vocabulary())))

1289 --->  ride
313 --->  wasnt
Vocabulary size: 10000


In [49]:
int_train_ds = train_ds.map(int_vectorize_text)
int_val_ds = val_ds.map(int_vectorize_text)
int_test_ds = test_ds.map(int_vectorize_text)

In [50]:
AUTOTUNE = tf.data.AUTOTUNE

def configure_dataset(dataset):
  return dataset.cache().prefetch(buffer_size=AUTOTUNE)

In [51]:
int_train_ds = configure_dataset(int_train_ds)
int_val_ds = configure_dataset(int_val_ds)
int_test_ds = configure_dataset(int_test_ds)

In [52]:
import os
import tensorflow as tf

base_dir = "/content/drive/MyDrive/Proyecto IA/Data"
os.makedirs(base_dir, exist_ok=True)

train_dir = os.path.join(base_dir, "train_dataset")
val_dir   = os.path.join(base_dir, "val_dataset")
test_dir  = os.path.join(base_dir, "test_dataset")

int_train_ds.save(train_dir)
int_val_ds.save(val_dir)
int_test_ds.save(test_dir)

print(f" Datasets guardados correctamente en:\n {base_dir}")

 Datasets guardados correctamente en:
 /content/drive/MyDrive/Proyecto IA/Data


In [53]:
vocab = int_vectorize_layer.get_vocabulary()
print("Tamaño del vocabulario:", len(vocab))


Tamaño del vocabulario: 10000


In [54]:
with open(os.path.join(base_dir, "vocab.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(vocab))
print("Vocabulario guardado en vocab.txt")

Vocabulario guardado en vocab.txt
